# Fine-tuning LayoutLMv3 pada CORD-v2 (untuk BAB IV)

Notebook ini melakukan *fine-tuning* `microsoft/layoutlmv3-base` untuk **token
classification** pada dataset **`naver-clova-ix/cord-v2`** (struk belanja),
lalu melaporkan F1 akhir dari **test split** yang benar-benar disisihkan.

Notebook ini **dibangun ulang** dari versi sebelumnya (yang mencapai F1 validasi
~0.9495) untuk memperbaiki empat isu metodologi:

1. **Test split disisihkan total.** Split validasi hanya dipakai untuk memilih
   *checkpoint* terbaik (`load_best_model_at_end`). Angka yang dilaporkan di
   BAB IV diambil **satu kali** dari test split di akhir (Bagian 7), bukan dari
   validasi.
2. **Kosakata label dari gabungan train + validation + test.** Mencegah kategori
   yang hanya muncul di val/test diam-diam dianggap `O`.
3. **Deteksi duplikat lintas ketiga split.** Gambar identik yang bocor dari
   train ke val/test dibuang dari split non-train, sehingga skor test tidak
   terinflasi oleh kebocoran.
4. **Demo inferensi memakai EasyOCR**, bukan Tesseract — konsisten dengan OCR
   resmi sistem di proposal.

Semua operasi acak memakai `seed = 42` secara eksplisit. Tiap tahap dibungkus
`try/except` bertanda supaya mudah tahu tahap mana yang gagal di Kaggle.


## 1. Lingkungan dan Instalasi

Satu sel instalasi dengan versi yang sudah terbukti kompatibel dengan Kaggle
(hasil beberapa kali *debugging*):

| Paket | Versi | Alasan |
| --- | --- | --- |
| `transformers` | `4.41.2` | LayoutLMv3 + `evaluation_strategy` (nama lama) masih stabil |
| `accelerate` | `0.34.2` | versi lama tak punya `clear_device_cache` yang dipakai `peft` |
| `datasets` | `3.6.0` | versi lama gagal konversi `Array2D`/`Array3D` (semantik `copy=False` NumPy 2.x) |
| `peft` | **di-uninstall** | notebook ini tidak pakai LoRA/PEFT; versi bawaan bentrok dengan `transformers==4.41.2` |
| `seqeval` | **tidak dipakai** | gagal install di Kaggle (`easy_install` sudah dihapus dari setuptools). Metrik BIO ditulis manual (Bagian 5) |
| `easyocr` | terbaru | hanya untuk sel demo inferensi paling akhir, bukan training |

Memakai `%pip` (magic command), bukan `!pip`, supaya masuk ke interpreter kernel
yang aktif.


In [ ]:
# === TAHAP: instalasi =======================================================
%pip uninstall -y -q peft seqeval
%pip install -q "transformers==4.41.2" "accelerate==0.34.2" "datasets==3.6.0" "tokenizers>=0.19,<0.20" "Pillow>=10.0.0" "sentencepiece" "easyocr"

print("Instalasi selesai. RESTART kernel TIDAK diperlukan di Kaggle bila memakai %pip.")


### 1b. Verifikasi mandiri lingkungan

Cek tiap modul dengan `importlib`. Kalau ada yang hilang atau versinya salah,
install ulang **paksa** lewat `sys.executable + subprocess` supaya benar-benar
masuk ke interpreter kernel yang aktif (bukan Python lain).


In [ ]:
import importlib
import subprocess
import sys

REQUIRED = {
    "transformers": "4.41.2",
    "accelerate": "0.34.2",
    "datasets": "3.6.0",
}
OPTIONAL = ["easyocr", "PIL", "numpy", "torch"]


def _pkg_version(mod_name: str) -> str:
    try:
        mod = importlib.import_module(mod_name)
        return getattr(mod, "__version__", "?")
    except Exception:
        return None


def _force_install(spec: str) -> None:
    print(f"  -> memasang ulang paksa: {spec}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", spec])


for mod_name, want in REQUIRED.items():
    have = _pkg_version(mod_name)
    if have is None:
        _force_install(f"{mod_name}=={want}")
    elif have != want:
        print(f"[warn] {mod_name} {have} != {want}")
        _force_install(f"{mod_name}=={want}")
    else:
        print(f"[ok]   {mod_name}=={have}")

# Muat ulang modul yang mungkin baru dipasang.
for mod_name in list(REQUIRED) + OPTIONAL:
    try:
        importlib.import_module(mod_name)
        print(f"[ok]   import {mod_name}")
    except Exception as e:  # noqa: BLE001
        print(f"[FAIL] import {mod_name}: {e}")

# peft harus benar-benar tidak ada.
if importlib.util.find_spec("peft") is not None:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "peft"])
    print("[ok]   peft di-uninstall ulang")
else:
    print("[ok]   peft tidak terpasang")


## 2. Memuat dan Membersihkan Data

- Muat `naver-clova-ix/cord-v2` langsung dari Hugging Face Hub: **train (800)**,
  **validation (100)**, **test (100)**.
- *Cleaning* konsisten ke ketiga split: gambar → RGB, *bounding box* di-*clip*
  agar tidak keluar batas gambar lalu dinormalisasi ke skala 0–1000, teks
  dibersihkan, contoh dengan kata terlalu sedikit dibuang, plus laporan
  statistik sebelum/sesudah.
- Deteksi duplikat pakai *image hashing* **lintas ketiga split sekaligus**;
  duplikat lintas split dibuang dari split yang **bukan** train.
- `seed = 42` eksplisit di semua operasi acak.


In [ ]:
# === TAHAP: setup umum + seed ==============================================
import os
import re
import json as _json
import random
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np

# Output notebook tidak berisik saat dibaca ulang.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", message=".*resume_download.*")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    HAS_GPU = torch.cuda.is_available()
except Exception:  # noqa: BLE001
    HAS_GPU = False

import transformers

transformers.set_seed(SEED)
print("transformers", transformers.__version__, "| GPU:", HAS_GPU)


In [ ]:
# === TAHAP: loading =========================================================
from datasets import load_dataset, DatasetDict

try:
    raw = load_dataset("naver-clova-ix/cord-v2")
    raw = DatasetDict(
        {
            "train": raw["train"],
            "validation": raw["validation"],
            "test": raw["test"],
        }
    )
except Exception as e:  # noqa: BLE001
    raise RuntimeError(f"[TAHAP loading] gagal memuat cord-v2: {e}") from e

for split, ds in raw.items():
    print(f"{split:11s}: {len(ds):4d} contoh | kolom: {ds.column_names}")


In [ ]:
# === TAHAP: parsing anotasi CORD-v2 ========================================
# ground_truth CORD-v2 = string JSON berisi 'valid_line'. Tiap 'valid_line' punya
# 'category' (mis. 'menu.nm', 'total.total_price') dan daftar 'words' dengan
# 'quad' (4 titik) + 'text'. Label kata = kategori barisnya, di-tag BIO
# (kata pertama B-, sisanya I-).


def _quad_to_box(quad):
    xs = [quad["x1"], quad["x2"], quad["x3"], quad["x4"]]
    ys = [quad["y1"], quad["y2"], quad["y3"], quad["y4"]]
    return [min(xs), min(ys), max(xs), max(ys)]


def parse_example(example):
    gt = _json.loads(example["ground_truth"])
    valid_lines = gt.get("valid_line", [])

    # Ukuran gambar: dari meta bila ada, kalau tidak dari PIL.
    meta = gt.get("meta", {}) or {}
    size = meta.get("image_size") or {}
    width = size.get("width") or example["image"].width
    height = size.get("height") or example["image"].height
    width = max(1, int(width))
    height = max(1, int(height))

    words, boxes, tags = [], [], []
    n_oob = 0  # bounding box di luar batas gambar (sebelum clip)

    for line in valid_lines:
        category = line.get("category", "other")
        line_words = line.get("words", [])
        first = True
        for w in line_words:
            text = (w.get("text") or "").strip()
            if not text:
                continue
            box = _quad_to_box(w["quad"])
            if box[0] < 0 or box[1] < 0 or box[2] > width or box[3] > height:
                n_oob += 1
            # clip ke batas gambar
            x0 = min(max(box[0], 0), width)
            y0 = min(max(box[1], 0), height)
            x1 = min(max(box[2], 0), width)
            y1 = min(max(box[3], 0), height)
            # normalisasi ke 0..1000 (skala yang diharapkan LayoutLMv3)
            nb = [
                int(1000 * x0 / width),
                int(1000 * y0 / height),
                int(1000 * x1 / width),
                int(1000 * y1 / height),
            ]
            nb = [min(max(v, 0), 1000) for v in nb]
            # jaga x0<=x1, y0<=y1
            nb = [min(nb[0], nb[2]), min(nb[1], nb[3]), max(nb[0], nb[2]), max(nb[1], nb[3])]

            words.append(text)
            boxes.append(nb)
            tags.append(("B-" if first else "I-") + category)
            first = False

    return {
        "words": words,
        "bboxes": boxes,
        "ner_tags_str": tags,
        "n_words": len(words),
        "n_oob": n_oob,
    }


parsed = raw.map(
    parse_example,
    remove_columns=[c for c in raw["train"].column_names if c != "image"],
    desc="parsing ground_truth",
)

tot_oob = sum(sum(parsed[s]["n_oob"]) for s in parsed)
print("Total bounding box di luar batas (di-clip):", tot_oob)
for s in parsed:
    wc = parsed[s]["n_words"]
    print(f"{s:11s}: kata/contoh min={min(wc)} max={max(wc)} rata2={np.mean(wc):.1f}")


In [ ]:
# === TAHAP: cleaning =======================================================
MIN_WORDS = 3  # buang contoh dengan kata terlalu sedikit

_ws_re = re.compile(r"\s+")


def clean_text_fields(example):
    words = [_ws_re.sub(" ", w).strip() for w in example["words"]]
    keep = [i for i, w in enumerate(words) if w != ""]
    return {
        "words": [words[i] for i in keep],
        "bboxes": [example["bboxes"][i] for i in keep],
        "ner_tags_str": [example["ner_tags_str"][i] for i in keep],
        "n_words": len(keep),
    }


before = {s: len(parsed[s]) for s in parsed}
cleaned = parsed.map(clean_text_fields, desc="membersihkan teks")
cleaned = cleaned.filter(lambda ex: ex["n_words"] >= MIN_WORDS, desc=f"filter >= {MIN_WORDS} kata")

# Normalisasi gambar ke RGB (lazy: dilakukan saat encoding juga, tapi kita
# pastikan mode-nya konsisten dengan memetakan sekali).
def _to_rgb(ex):
    img = ex["image"]
    if img.mode != "RGB":
        ex["image"] = img.convert("RGB")
    return ex


cleaned = cleaned.map(_to_rgb, desc="normalisasi gambar -> RGB")

print("Statistik sebelum/sesudah cleaning:")
for s in cleaned:
    print(f"  {s:11s}: {before[s]:4d} -> {len(cleaned[s]):4d}  (-{before[s] - len(cleaned[s])})")


In [ ]:
# === TAHAP: deteksi duplikat LINTAS split ==================================
# Hash gambar (grayscale 32x32) untuk semua contoh di ketiga split sekaligus.
# Kalau satu hash muncul di >1 split: pertahankan di train bila ada di train,
# selain itu pertahankan di validation; buang dari split lainnya.


def _img_hash(img):
    g = img.convert("L").resize((32, 32))
    return hashlib.md5(np.asarray(g, dtype=np.uint8).tobytes()).hexdigest()


hash_to_locs = defaultdict(list)  # hash -> [(split, idx), ...]
for s in cleaned:
    for i, img in enumerate(cleaned[s]["image"]):
        hash_to_locs[_img_hash(img)].append((s, i))

PRIORITY = {"train": 0, "validation": 1, "test": 2}
drop = {s: set() for s in cleaned}
cross_split_dups = 0
within_split_dups = 0

for h, locs in hash_to_locs.items():
    if len(locs) == 1:
        continue
    splits_present = {s for s, _ in locs}
    if len(splits_present) > 1:
        cross_split_dups += 1
        keeper_split = min(splits_present, key=lambda s: PRIORITY[s])
        for s, i in locs:
            if s != keeper_split:
                drop[s].add(i)
        # kalau keeper_split punya >1 salinan, sisakan satu
        keep_idxs = [i for s, i in locs if s == keeper_split]
        for i in keep_idxs[1:]:
            drop[keeper_split].add(i)
    else:
        within_split_dups += 1
        s = next(iter(splits_present))
        idxs = sorted(i for _, i in locs)
        for i in idxs[1:]:
            drop[s].add(i)

print(f"Kelompok hash duplikat lintas split : {cross_split_dups}")
print(f"Kelompok hash duplikat dalam 1 split: {within_split_dups}")

from datasets import DatasetDict as _DD

deduped = _DD()
for s in cleaned:
    keep = [i for i in range(len(cleaned[s])) if i not in drop[s]]
    deduped[s] = cleaned[s].select(keep)
    print(f"  {s:11s}: {len(cleaned[s]):4d} -> {len(deduped[s]):4d}  (buang {len(drop[s])})")


## 3. Kosakata Label

Daftar label BIO dibangun dari kategori yang muncul di **gabungan train +
validation + test**, bukan hanya train. `label2id` / `id2label` disimpan
eksplisit, dan jumlah kemunculan tiap label per split dicetak untuk verifikasi.

> **Catatan untuk BAB IV.** Seluruh data CORD-v2 berasal dari `valid_line` yang
> sudah teranotasi sebagai entitas, sehingga **tidak ada contoh training murni
> untuk kelas `O`** (bukan-entitas). Ini celah *training-vs-inferensi* yang harus
> disebut di BAB IV: pada inferensi struk nyata, teks non-entitas (footer, nomor
> telepon, dsb.) tidak pernah dilihat model sebagai `O` saat training. Ini
> keterbatasan sumber data, bukan sesuatu yang bisa diperbaiki dari sisi
> pipeline training.


In [ ]:
# === TAHAP: kosakata label =================================================
label_counter = {s: Counter() for s in deduped}
all_labels = set()
for s in deduped:
    for tags in deduped[s]["ner_tags_str"]:
        label_counter[s].update(tags)
        all_labels.update(tags)

# Urutan stabil: 'O' dulu, lalu sisanya alfabetis. 'O' tetap dimasukkan ke
# kosakata supaya model bisa memprediksinya saat inferensi (lihat catatan di atas).
label_list = ["O"] + sorted(all_labels)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Jumlah label (termasuk 'O'): {len(label_list)}\n")
header = f"{'label':32s} " + " ".join(f"{s[:5]:>6s}" for s in deduped)
print(header)
print("-" * len(header))
for l in label_list:
    counts = " ".join(f"{label_counter[s][l]:6d}" for s in deduped)
    print(f"{l:32s} {counts}")

missing_in_train = [l for l in label_list if l != "O" and label_counter["train"][l] == 0]
if missing_in_train:
    print("\n[PENTING untuk BAB IV] label tanpa contoh di TRAIN (hanya di val/test):")
    for l in missing_in_train:
        print("  -", l)
else:
    print("\nSemua label entitas punya minimal 1 contoh di train.")


## 4. Encoding

- `LayoutLMv3Processor` dengan `apply_ocr=False` — CORD-v2 sudah punya kata +
  *bounding box* dari anotasi, tidak perlu OCR ulang saat training.
- `pixel_values` dikonversi **eksplisit dari tensor ke list** sebelum disimpan
  ke kolom dataset, supaya tidak gagal serialisasi ke `Array3D`.
- Diterapkan ke ketiga split.


In [ ]:
# === TAHAP: encoding ======================================================
from transformers import LayoutLMv3Processor
from datasets import Features, Sequence, Value, Array2D, Array3D

MODEL_CHECKPOINT = "microsoft/layoutlmv3-base"
MAX_LEN = 512

processor = LayoutLMv3Processor.from_pretrained(MODEL_CHECKPOINT, apply_ocr=False)


def encode_example(ex):
    image = ex["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    word_labels = [label2id[t] for t in ex["ner_tags_str"]]

    enc = processor(
        image,
        ex["words"],
        boxes=ex["bboxes"],
        word_labels=word_labels,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    # buang dimensi batch (1, ...) dan konversi pixel_values tensor -> list
    # (list eksplisit mencegah gagal serialisasi ke Array3D di datasets).
    return {
        "input_ids": enc["input_ids"][0].numpy().astype("int64"),
        "attention_mask": enc["attention_mask"][0].numpy().astype("int64"),
        "bbox": enc["bbox"][0].numpy().astype("int64"),
        "pixel_values": enc["pixel_values"][0].float().numpy().astype("float32").tolist(),
        "labels": enc["labels"][0].numpy().astype("int64"),
    }


features = Features(
    {
        "input_ids": Sequence(Value("int64")),
        "attention_mask": Sequence(Value("int64")),
        "bbox": Array2D(shape=(MAX_LEN, 4), dtype="int64"),
        "pixel_values": Array3D(shape=(3, 224, 224), dtype="float32"),
        "labels": Sequence(Value("int64")),
    }
)

encoded = deduped.map(
    encode_example,
    remove_columns=deduped["train"].column_names,
    features=features,
    desc="encoding LayoutLMv3",
)
encoded.set_format("torch")

print(encoded)
sample = encoded["train"][0]
print({k: tuple(v.shape) for k, v in sample.items()})


## 5. Metrik Evaluasi Kustom (BIO, tanpa `seqeval`)

Implementasi manual precision / recall / F1 **berbasis entitas** (span BIO):
sebuah entitas dianggap benar hanya bila tipe, posisi awal, dan posisi akhirnya
persis sama dengan *ground truth*. Dipanggil `Trainer` lewat `compute_metrics`.


In [ ]:
# === TAHAP: metrik BIO manual =============================================
def _bio_spans(tags):
    # Ubah urutan tag BIO -> himpunan span (tipe, start, end_exclusive).
    spans = set()
    cur_type, start = None, None
    for i, tag in enumerate(tags):
        if tag == "O":
            if cur_type is not None:
                spans.add((cur_type, start, i))
                cur_type, start = None, None
        elif tag.startswith("B-"):
            if cur_type is not None:
                spans.add((cur_type, start, i))
            cur_type, start = tag[2:], i
        elif tag.startswith("I-"):
            t = tag[2:]
            if cur_type is None:
                cur_type, start = t, i          # I- tanpa B- di depan: mulai span
            elif t != cur_type:
                spans.add((cur_type, start, i))  # ganti tipe: tutup lalu buka
                cur_type, start = t, i
        else:
            if cur_type is not None:
                spans.add((cur_type, start, i))
            cur_type, start = None, None
    if cur_type is not None:
        spans.add((cur_type, start, len(tags)))
    return spans


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    tp = fp = fn = 0
    tok_correct = tok_total = 0

    for pred_row, label_row in zip(preds, labels):
        pred_tags, gold_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            pred_tags.append(id2label[int(p)])
            gold_tags.append(id2label[int(l)])
            tok_total += 1
            tok_correct += int(p == l)

        pred_spans = _bio_spans(pred_tags)
        gold_spans = _bio_spans(gold_tags)
        tp += len(pred_spans & gold_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "token_accuracy": tok_correct / tok_total if tok_total else 0.0,
    }


# sanity check kecil
_demo = ["B-menu.nm", "I-menu.nm", "O", "B-total.total_price"]
assert _bio_spans(_demo) == {("menu.nm", 0, 2), ("total.total_price", 3, 4)}
print("compute_metrics siap.")


## 6. Model dan Training

- `LayoutLMv3ForTokenClassification` dari `microsoft/layoutlmv3-base` dengan
  `num_labels` = jumlah label dari Bagian 3.
- `Trainer` dilatih dengan `train_dataset` = train, `eval_dataset` =
  **validation saja**. Test split **tidak disentuh** di tahap ini.


In [ ]:
# === TAHAP: bangun model + Trainer ========================================
from transformers import (
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

OUTPUT_DIR = "/kaggle/working/layoutlmv3-cord-v2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    fp16=HAS_GPU,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],   # HANYA validation
    tokenizer=processor,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer siap. Train:", len(encoded["train"]), "| Eval(val):", len(encoded["validation"]))


In [ ]:
# === TAHAP: training ======================================================
try:
    train_result = trainer.train()
    trainer.save_metrics("train", train_result.metrics)
    print("\nMetrik training:", train_result.metrics)
    print("\nEvaluasi VALIDATION (checkpoint terbaik, dipakai untuk pemilihan model):")
    val_metrics = trainer.evaluate(eval_dataset=encoded["validation"])
    for k, v in val_metrics.items():
        print(f"  {k}: {v}")
except Exception as e:  # noqa: BLE001
    raise RuntimeError(f"[TAHAP training] gagal: {e}") from e


## 7. Hasil Resmi untuk Laporan BAB IV (dari Test Split, Bukan Validation)

Sel di bawah menjalankan `trainer.evaluate()` **satu kali** pada **test split**
yang belum pernah disentuh selama training maupun pemilihan *checkpoint*.

**Angka inilah yang dilaporkan di BAB IV.** Angka validasi di Bagian 6 hanya
untuk memilih *checkpoint* terbaik dan tidak boleh dilaporkan sebagai hasil akhir.


In [ ]:
# === TAHAP: evaluasi akhir (SEKALI, di test split) ========================
try:
    test_metrics = trainer.evaluate(eval_dataset=encoded["test"], metric_key_prefix="test")
except Exception as e:  # noqa: BLE001
    raise RuntimeError(f"[TAHAP evaluasi akhir] gagal: {e}") from e

trainer.save_metrics("test", test_metrics)

print("=" * 60)
print("HASIL RESMI BAB IV  --  TEST SPLIT CORD-v2 (n = {})".format(len(encoded["test"])))
print("=" * 60)
for k in ("test_precision", "test_recall", "test_f1", "test_token_accuracy", "test_loss"):
    if k in test_metrics:
        print(f"  {k:22s}: {test_metrics[k]:.4f}")
print("=" * 60)


In [ ]:
# === Rincian F1 per kategori entitas (untuk tabel di BAB IV) ==============
from collections import defaultdict as _dd

pred_out = trainer.predict(encoded["test"])
_logits, _labels = pred_out.predictions, pred_out.label_ids
_preds = np.argmax(_logits, axis=2)

per_type = _dd(lambda: [0, 0, 0])  # tipe -> [tp, fp, fn]
for pr, la in zip(_preds, _labels):
    pt, gt = [], []
    for p, l in zip(pr, la):
        if l == -100:
            continue
        pt.append(id2label[int(p)])
        gt.append(id2label[int(l)])
    ps, gs = _bio_spans(pt), _bio_spans(gt)
    for span in ps & gs:
        per_type[span[0]][0] += 1
    for span in ps - gs:
        per_type[span[0]][1] += 1
    for span in gs - ps:
        per_type[span[0]][2] += 1

print(f"{'kategori':28s} {'P':>7s} {'R':>7s} {'F1':>7s} {'support':>8s}")
print("-" * 62)
for t in sorted(per_type):
    tp, fp, fn = per_type[t]
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    print(f"{t:28s} {p:7.3f} {r:7.3f} {f:7.3f} {tp + fn:8d}")


## 8. Simpan Model

Simpan model + processor ke `/kaggle/working/`. `push_to_hub()` dibiarkan
sebagai opsi (perlu token HF) untuk deployment ke Hugging Face Spaces.


In [ ]:
# === TAHAP: simpan model =================================================
SAVE_DIR = "/kaggle/working/layoutlmv3-cord-v2-final"
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

# simpan pemetaan label eksplisit sebagai artefak terpisah
with open(os.path.join(SAVE_DIR, "label_mapping.json"), "w") as f:
    _json.dump({"label_list": label_list, "label2id": label2id, "id2label": id2label}, f, indent=2)

print("Tersimpan ke:", SAVE_DIR)
print(sorted(os.listdir(SAVE_DIR)))

# --- Opsional: unggah ke Hugging Face Hub untuk deployment ke HF Spaces ---
# from huggingface_hub import login
# login(token="hf_xxx")   # atau set Kaggle Secret HF_TOKEN
# trainer.push_to_hub("username/layoutlmv3-cord-v2")
# processor.push_to_hub("username/layoutlmv3-cord-v2")


## 9. Demo Inferensi dengan EasyOCR (sanity check, bukan evaluasi resmi)

Unggah foto struk sendiri → **EasyOCR** (bukan Tesseract) ambil kata + *bounding
box* → model hasil *fine-tuning* memberi label entitas → sel ini **merekonstruksi
struk terstruktur** (daftar item + subtotal/pajak/diskon/total).

Cara pakai: di panel kanan Kaggle **Add Input → Upload** foto struk (jpg/png),
atau taruh file di `/kaggle/working/`. Sel ini otomatis mencari gambar di sana.

Sel ini **mandiri** — kalau kernel sudah di-restart, model + processor dimuat
ulang dari `/kaggle/working/layoutlmv3-cord-v2-final`. Ini hanya *sanity check*;
**evaluasi resmi BAB IV tetap dari test split CORD-v2 di Bagian 7.**


In [ ]:
# === TAHAP: demo inferensi EasyOCR + rekonstruksi struk ==================
import glob
from PIL import Image

try:
    import torch  # noqa: F401
except Exception:
    pass

# --- 0. Pastikan model + processor + id2label tersedia (mandiri) ----------
_FINAL_DIR = "/kaggle/working/layoutlmv3-cord-v2-final"
if "model" not in globals() or "processor" not in globals():
    from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor

    print("Memuat model dari", _FINAL_DIR)
    model = LayoutLMv3ForTokenClassification.from_pretrained(_FINAL_DIR)
    processor = LayoutLMv3Processor.from_pretrained(_FINAL_DIR, apply_ocr=False)

if "id2label" not in globals():
    with open(os.path.join(_FINAL_DIR, "label_mapping.json")) as f:
        _m = _json.load(f)
    id2label = {int(k): v for k, v in _m["id2label"].items()}

_MAX_LEN = globals().get("MAX_LEN", 512)
_GPU = globals().get("HAS_GPU", False)

# --- 1. Cari gambar struk ------------------------------------------------
IMAGE_PATH = "/kaggle/input/struk-demo/struk.jpg"   # <-- boleh disesuaikan manual
if not os.path.exists(IMAGE_PATH):
    cand = []
    for ext in ("jpg", "jpeg", "png", "webp"):
        cand += glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)
        cand += glob.glob(f"/kaggle/working/*.{ext}")
    IMAGE_PATH = cand[0] if cand else None


def _entities_from_words(words, labels):
    # Gabung kata berurutan berlabel sama (abaikan prefix B-/I-) jadi entitas.
    ents, cur_field, cur_words = [], None, []
    for w, lab in zip(words, labels):
        field = lab[2:] if lab[:2] in ("B-", "I-") else ("" if lab == "O" else lab)
        is_b = lab.startswith("B-")
        if field == "" :
            if cur_field:
                ents.append((cur_field, " ".join(cur_words)))
            cur_field, cur_words = None, []
        elif field == cur_field and not is_b:
            cur_words.append(w)
        else:
            if cur_field:
                ents.append((cur_field, " ".join(cur_words)))
            cur_field, cur_words = field, [w]
    if cur_field:
        ents.append((cur_field, " ".join(cur_words)))
    return ents


if IMAGE_PATH is None:
    print("Tidak ada gambar struk ditemukan. Upload lewat 'Add Input' atau ke "
          "/kaggle/working lalu jalankan ulang sel ini.")
else:
    try:
        import easyocr

        image = Image.open(IMAGE_PATH).convert("RGB")
        W, H = image.size
        print("Gambar:", IMAGE_PATH, f"({W}x{H})")

        reader = easyocr.Reader(["en", "id"], gpu=_GPU)
        ocr = reader.readtext(np.asarray(image))
        # urutkan baca: atas->bawah, lalu kiri->kanan
        ocr.sort(key=lambda r: (min(p[1] for p in r[0]), min(p[0] for p in r[0])))

        words, boxes, confs = [], [], []
        for box, text, conf in ocr:
            text = (text or "").strip()
            if not text:
                continue
            xs = [p[0] for p in box]
            ys = [p[1] for p in box]
            nb = [
                int(1000 * min(xs) / W), int(1000 * min(ys) / H),
                int(1000 * max(xs) / W), int(1000 * max(ys) / H),
            ]
            nb = [min(max(v, 0), 1000) for v in nb]
            words.append(text)
            boxes.append([min(nb[0], nb[2]), min(nb[1], nb[3]), max(nb[0], nb[2]), max(nb[1], nb[3])])
            confs.append(float(conf))
        print(f"EasyOCR: {len(words)} kata (rata2 conf {np.mean(confs):.2f})\n")

        enc = processor(
            image, words, boxes=boxes,
            truncation=True, padding="max_length", max_length=_MAX_LEN,
            return_tensors="pt",
        )
        model.eval()
        with torch.no_grad():
            out = model(**{k: v.to(model.device) for k, v in enc.items()})
        pred_ids = out.logits.argmax(-1)[0].tolist()

        # label per KATA (ambil sub-token pertama tiap kata)
        word_ids = enc.word_ids(0)
        word_labels = [None] * len(words)
        for tok_idx, wid in enumerate(word_ids):
            if wid is not None and word_labels[wid] is None:
                word_labels[wid] = id2label[pred_ids[tok_idx]]
        word_labels = [l or "O" for l in word_labels]

        print("--- Prediksi mentah (kata -> label) ---")
        for w, l in zip(words, word_labels):
            if l != "O":
                print(f"  {w[:28]:28s} -> {l}")

        # --- 2. Rekonstruksi struk terstruktur ---
        ents = _entities_from_words(words, word_labels)
        items, cur = [], {}
        totals = {}
        for field, text in ents:
            if field == "menu.nm":
                if cur.get("nama"):
                    items.append(cur)
                    cur = {}
                cur["nama"] = text
            elif field in ("menu.cnt", "menu.num"):
                cur["qty"] = text
            elif field in ("menu.unitprice", "menu.sub.unitprice"):
                cur["harga_satuan"] = text
            elif field in ("menu.price", "menu.itemsubtotal", "menu.sub.price"):
                cur["subtotal"] = text
            elif field in ("menu.discountprice",):
                cur["diskon_item"] = text
            elif field.startswith(("sub_total.", "total.")):
                totals[field] = text
        if cur.get("nama"):
            items.append(cur)

        print("\n" + "=" * 52)
        print("REKONSTRUKSI STRUK")
        print("=" * 52)
        for it in items:
            print(f"  - {it.get('nama','?'):30s} "
                  f"x{it.get('qty','?'):>3s}  {it.get('subtotal', it.get('harga_satuan','?')):>12s}")
        if not items:
            print("  (tidak ada baris item terdeteksi — cek kualitas foto / OCR)")
        print("-" * 52)
        for k in sorted(totals):
            print(f"  {k:28s} {totals[k]:>12s}")

        struk_json = {"items": items, "ringkasan": totals, "sumber_gambar": os.path.basename(IMAGE_PATH)}
        print("\n--- JSON (bentuk yang nanti dikembalikan HF Space) ---")
        print(_json.dumps(struk_json, indent=2, ensure_ascii=False))
    except Exception as e:  # noqa: BLE001
        import traceback

        print(f"[TAHAP demo inferensi] gagal (tidak fatal): {e}")
        traceback.print_exc()


## 10. Catatan Metodologi (ringkas, untuk penulisan BAB IV)

- **Angka yang dilaporkan** = F1 / precision / recall entitas dari **test split
  CORD-v2** (Bagian 7), dihitung sekali setelah *checkpoint* terbaik dipilih
  berdasarkan **validation**. Tidak ada kebocoran pemilihan model ke angka akhir.
- **Metrik berbasis entitas (span BIO)**, bukan token — sebuah entitas benar
  hanya bila tipe + batas span persis sama. Implementasi manual (Bagian 5),
  tanpa `seqeval`.
- **Kosakata label** dibangun dari gabungan ketiga split; label yang hanya
  muncul di val/test dilaporkan eksplisit di Bagian 3.
- **Deduplikasi lintas split** dengan *image hash* — duplikat dibuang dari split
  non-train, jumlahnya dilaporkan di Bagian 2.
- **Kelas `O`**: tidak ada contoh training murni (semua data CORD-v2 = entitas
  ber-anotasi). Sebutkan sebagai keterbatasan *training-vs-inferensi* di BAB IV.
- **OCR sistem = EasyOCR** (Bagian 9), konsisten dengan proposal; Tesseract
  tidak dipakai di mana pun.
- **Reproduksibilitas**: `seed = 42` di `random`, `numpy`, `torch`,
  `transformers.set_seed`, dan `TrainingArguments(seed, data_seed)`.
